<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c05-header.png" alt="Nextia Learning · Machine Learning: From Problem to Reliable Model" width="100%">

# Estimate house prices

**[Estimate house prices](https://learning.nextia-ai.com/courses/ml/m09/estimate-house-prices/)** · Machine Learning: From Problem to Reliable Model · Module 9, case study 2 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will** estimate the sale price of a house from its size, quality and place, for a lender's first valuation. You go from the business question to a tested model, an experiment log, error slices, one final test and a recommendation.

| | |
|---|---|
| **Time** | About 2 hours with the lesson page. The notebook runs in about 2 minutes. |
| **Needs** | An internet connection for the setup cells. No account. The first cell installs pandas 3.0.6 and scikit-learn 1.9.1. |
| **You should know** | Modules 1 to 8 of this course, especially [Train, validation and test sets](https://learning.nextia-ai.com/courses/ml/m03/train-validation-and-test-sets/), [Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/), [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/) and [Slice analysis](https://learning.nextia-ai.com/courses/ml/m07/slice-analysis/). |
| **You do not need** | The local ticket-model project. This notebook works in its own folder, `house-prices/`. |
| **Tested** | Python 3.14.6 with pandas 3.0.6, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on macOS, from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/estimate-house-prices/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C05/M09-L02-estimate-house-prices/estimate-house-prices-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs pandas 3.0.6 and scikit-learn 1.9.1, the versions that the course uses, and matplotlib for the charts. It needs an internet connection and takes up to a minute the first time.

You should see `pandas 3.0.6` and `scikit-learn 1.9.1`. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.0.6 scikit-learn==1.9.1 matplotlib
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the data

Run the next cell. It downloads `king_county_house_sales.csv.gz` (0.72 MB) from the Nextia labs repository into a folder `house-prices`, and checks its checksum. It needs an internet connection and no account, and takes a few seconds.

You should see `Ready: king_county_house_sales.csv.gz, checksum OK.` If you run the cell again, it uses the file that is already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings (it needs a verified phone number).
- `wrong checksum`: delete the folder `house-prices` (in Colab: the **Files** panel on the left), then run the cell again.

**The data.** *House Sales in King County, USA* (house_sales), sales from May 2014 to May 2015. OpenML dataset 42092, version 2, https://www.openml.org/d/42092, first published on Kaggle by harlfoxem (https://www.kaggle.com/datasets/harlfoxem/housesalesprediction). Licence: [CC0 1.0 Public Domain](https://creativecommons.org/publicdomain/zero/1.0/), as stated on both pages. The sales are public records of King County, Washington. We host the file as a gzip CSV, with the date written as `YYYY-MM-DD`; nothing else is changed.

In [ ]:
# Setup: download the case-study data into house-prices/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

DATA_URL = "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C05/case-studies/data/"
NAME, SHA256 = "king_county_house_sales.csv.gz", "0e2fd1937fbc724622e00fefebe060e28587882725663050c5c25fef74ce193c"
HERE = Path.cwd()
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("house-prices").resolve()
WORK.mkdir(exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

path = WORK / NAME
if not (path.exists() and sha256(path) == SHA256):
    if (HERE / NAME).exists():  # a copy next to the notebook
        shutil.copy(HERE / NAME, path)
    else:
        try:
            urllib.request.urlretrieve(DATA_URL + NAME, path)
        except OSError:  # Python without certificates (python.org on macOS): try curl
            if subprocess.run(["curl", "-fsSL", "-o", str(path), DATA_URL + NAME]).returncode:
                raise SystemExit(f"Could not download {DATA_URL + NAME}. Check your internet connection.")
    if sha256(path) != SHA256:
        raise SystemExit(f"{NAME} has the wrong checksum. Delete the folder house-prices and run again.")
os.chdir(WORK)  # the experiment log and the saved model go here too
print(f"Ready: {NAME}, checksum OK.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd` and numpy as `np`, and sets pandas to show 4 decimals. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The situation

Nadia Haddad leads the valuation team of a regional mortgage lender in the Seattle area. Before a loan is approved, a human appraiser values the house, which takes days. Nadia wants a **first estimate** on the day of the application, to spot cases where the agreed price looks far from the value. Her target: the estimate is **within ±10% of the sale price** for most houses. Samir Patel, the analyst, builds it. The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/estimate-house-prices/#the-situation) tells the full story.

## 2. The data

Run the next cell. It loads the file and shows the first three sales.

In [ ]:
houses = pd.read_csv("king_county_house_sales.csv.gz", parse_dates=["date"])
print(houses.shape)
print("sales from", houses["date"].min().date(), "to", houses["date"].max().date())
print(houses[["date", "price", "bedrooms", "bathrooms", "sqft_living", "grade", "zipcode"]].head(3).to_string())

> **Check.** You should see `(21613, 21)` and sales from `2014-05-02` to `2015-05-27`.

**What this tells us.** One row is one sale of one house: 21,613 sales in about 13 months. `price` is the target, in US dollars.

Now the first checks. Run the cell.

In [ ]:
print("missing values:", int(houses.isna().sum().sum()), "| duplicate rows:", int(houses.duplicated().sum()))
print("houses sold more than once:", int(houses["id"].duplicated().sum()), "extra sales")
print("bedrooms:", houses["bedrooms"].value_counts().sort_index().to_dict())
print("bathrooms 0:", int((houses["bathrooms"] == 0).sum()),
      "| yr_renovated 0:", int((houses["yr_renovated"] == 0).sum()),
      "| sqft_basement 0:", int((houses["sqft_basement"] == 0).sum()))

> **Check.** You should see no missing values, `houses sold more than once: 177 extra sales`, 13 houses with 0 bedrooms, one with 33, and `bathrooms 0: 10`.

**What this tells us.** A 0 in `yr_renovated` means "never renovated" (20,699 houses), and a 0 in `sqft_basement` means "no basement". They are codes, not measurements. One house has 33 bedrooms. Look at it next to the other large houses. Run the cell.

In [ ]:
print(houses.loc[houses["bedrooms"] >= 10, ["bedrooms", "bathrooms", "sqft_living", "price"]].to_string())

> **Check.** You should see the row with 33 bedrooms, 1.75 bathrooms and 1,620 sq ft.

**What this tells us.** 33 bedrooms in 1,620 sq ft is not possible: it is a typing error for 3. Fix it, and write the fix down. Run the cell.

In [ ]:
houses.loc[houses["bedrooms"] == 33, "bedrooms"] = 3  # 1,620 sq ft: a typing error for 3
print("largest number of bedrooms now:", int(houses["bedrooms"].max()))

> **Check.** You should see `largest number of bedrooms now: 11`.

Some houses were sold more than once. How much did the price change? Run the cell.

In [ ]:
repeat = houses[houses["id"].duplicated(keep=False)].sort_values(["id", "date"])
first, last = repeat.groupby("id")["price"].first(), repeat.groupby("id")["price"].last()
print("houses:", repeat["id"].nunique(), "| median change from first to last sale:",
      f"{100 * ((last / first).median() - 1):.1f}%")

> **Check.** You should see `houses: 176 | median change from first to last sale: 54.4%`.

**What this tells us.** Most repeat sales in one year are houses bought, renovated and sold again. The data still shows the old description, so the model cannot see the work. Two sales of one house are not independent examples: they must stay in the same part of the split.

Now the price. Run the cell; the chart after it shows the price and its logarithm.

In [ ]:
print(houses["price"].describe().round(0).to_dict())
print("share above 1 million:", round(float((houses["price"] > 1_000_000).mean()), 3))

> **Check.** You should see a median of 450,000, a mean of 540,088 and a maximum of 7,700,000. About 6.8% of sales are above 1 million.

**What this tells us.** The price is **skewed**: many ordinary houses and a long tail of expensive ones. A miss of 50,000 dollars is large for a 250,000-dollar house and small for a 2-million-dollar house. So we measure errors in **percent**, and the models learn the **logarithm** of the price. Run the chart cell.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
houses["price"].plot.hist(bins=60, ax=axes[0])
axes[0].set_xlabel("Price (US dollars)")
np.log10(houses["price"]).plot.hist(bins=60, ax=axes[1])
axes[1].set_xlabel("log10 of the price")
plt.show()

Prices move with the market. Run the cell to see the sales and the median price per month.

In [ ]:
months = houses.groupby(houses["date"].dt.to_period("M"))["price"].agg(sales="size", median_price="median")
print(months.T.to_string())

> **Check.** You should see fewer sales in winter (978 in January 2015) and a median of 476,500 in April 2015, the highest.

**What this tells us.** Spring is busy and expensive. A model that learns in winter will be too low in spring.

The place matters most in house prices. The next cell puts each sale in one of three areas, and prints the median price per square foot. Run it.

In [ ]:
def area(part):
    """Three areas: zip codes 981xx (Seattle and close neighbours), then north/east and south."""
    seattle = part["zipcode"] // 100 == 981
    return np.select([seattle, part["lat"] >= 47.5], ["Seattle", "North and east"], "South")

houses["area"] = area(houses)
per_sqft = (houses["price"] / houses["sqft_living"]).groupby(houses["area"]).median().round(0)
print(houses["area"].value_counts().to_dict())
print("median price per sq ft:", per_sqft.to_dict())

> **Check.** You should see 8,977 sales in Seattle (median 291 dollars per sq ft), 7,209 in the north and east (264) and 5,427 in the south (165).

## 3. Split and baseline

Split by time, and keep all sales of a house in the part of its **first** sale: **train** before 2015, **valid** January and February 2015, **test** March to May 2015. Run the cell.

In [ ]:
TARGET = "price"
FEATURES = ["bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors", "waterfront", "view", "condition",
            "grade", "sqft_above", "sqft_basement", "yr_built", "yr_renovated", "lat", "long",
            "sqft_living15", "sqft_lot15", "zipcode"]
first_sale = houses.groupby("id")["date"].transform("min")  # every sale of a house goes where its first sale goes
train = houses[first_sale < "2015-01-01"]
valid = houses[(first_sale >= "2015-01-01") & (first_sale < "2015-03-01")]
test = houses[first_sale >= "2015-03-01"]
print("train", len(train), "| valid", len(valid), "| test", len(test))
print("sales moved to an earlier part:", int((houses["date"] != first_sale).sum()))

> **Check.** You should see `train 14768 | valid 2191 | test 4654` and `sales moved to an earlier part: 177`.

The next cell defines `report()`. It gives the **median error** in % (positive: the estimate is too low), the **MAPE** from [Regression metrics](https://learning.nextia-ai.com/courses/ml/m03/regression-metrics/), the share **within 10%** (Nadia's target) and the share **over by 10% or more** (the risk for a lender). Run it. It prints nothing.

In [ ]:
def report(real, estimate):
    """Errors as percentages of the real price (positive = the estimate is too low)."""
    pct = 100 * (real - estimate) / real
    return {"median error %": round(float(np.median(pct)), 1),
            "MAPE %": round(float(np.mean(np.abs(pct))), 1),
            "within 10%": round(float(np.mean(np.abs(pct) <= 10)), 3),
            "over by 10%+": round(float(np.mean(pct < -10)), 3)}

Three baselines, each learned from train only: the median price; the median price per square foot × the living area; and the median price per square foot **in the same zip code** × the living area. The last one is what a valuer does on the back of an envelope.

> **Predict.** Which baseline gets most estimates within 10%? Then run the cell.

In [ ]:
median_estimate = np.full(len(valid), train[TARGET].median())
per_sqft = (train[TARGET] / train["sqft_living"]).median()
zip_per_sqft = (train[TARGET] / train["sqft_living"]).groupby(train["zipcode"]).median()
baselines = {"median price": median_estimate,
             "price per sq ft": per_sqft * valid["sqft_living"],
             "zip price per sq ft": valid["zipcode"].map(zip_per_sqft) * valid["sqft_living"]}
baseline_results = {name: report(valid[TARGET], estimate) for name, estimate in baselines.items()}
print(pd.DataFrame(baseline_results).T.to_string())

> **Check.** For the zip-code baseline you should see a MAPE of 19.2 and 0.372 within 10%.

**What this tells us.** Size and place together already get 37% of houses within 10%. Any model must beat that.

### The experiment log

The next cell defines `log_experiment()` and `show_log()`, as in [Machine learning is iterative](https://learning.nextia-ai.com/courses/ml/m05/machine-learning-is-iterative/#the-experiment-log). The columns are the CV MAPE on train (mean and standard deviation of 5 folds), the MAPE on valid and the share within 10% on valid. Run it. It prints nothing.

In [ ]:
import datetime
from pathlib import Path

LOG_FILE = Path("experiment_log.csv")
LOG_COLUMNS = ["id", "date", "change", "cv_mape", "cv_std", "valid_mape", "valid_within_10", "decision", "reason"]

def log_experiment(id, change, cv, result, decision, reason):
    """Add one row (cv = (mean, std) or None), or replace the row with the same id."""
    log = pd.read_csv(LOG_FILE) if LOG_FILE.exists() else pd.DataFrame(columns=LOG_COLUMNS)
    row = {"id": id, "date": datetime.date.today().isoformat(), "change": change,
           "cv_mape": cv[0] if cv else None, "cv_std": cv[1] if cv else None,
           "valid_mape": result["MAPE %"], "valid_within_10": result["within 10%"],
           "decision": decision, "reason": reason}
    log = pd.concat([log[log["id"] != id], pd.DataFrame([row])], ignore_index=True)
    log.sort_values("id").to_csv(LOG_FILE, index=False)

def show_log():
    log = pd.read_csv(LOG_FILE)
    print(log[["id", "change", "cv_mape", "cv_std", "valid_mape", "valid_within_10", "decision"]].to_string(index=False))

LOG_FILE.unlink(missing_ok=True)  # start a new log

Log the three baselines. Baselines have nothing to tune, so they have no CV score. Run the cell.

In [ ]:
log_experiment("E01", "Baseline: the median price of train", None, baseline_results["median price"],
               "baseline", "The floor.")
log_experiment("E02", "Baseline: median price per sq ft x living area", None, baseline_results["price per sq ft"],
               "baseline", "Size alone.")
log_experiment("E03", "Baseline: median price per sq ft in the zip code x living area", None,
               baseline_results["zip price per sq ft"], "baseline", "Size and place: what a quick desk estimate does.")
show_log()

## 4. Models

Models are compared with 5-fold cross-validation on train, as in [Tune hyperparameters](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/). `GroupKFold` keeps all sales of one house in one fold. Run the cell. It prints nothing.

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_score

def cv_mape(model):
    """5-fold CV on train, the sales of one house in one fold; mean and std of the MAPE, in %."""
    scores = -cross_val_score(model, train, train[TARGET], groups=train["id"], cv=GroupKFold(n_splits=5),
                              scoring="neg_mean_absolute_percentage_error")
    return round(float(100 * scores.mean()), 2), round(float(100 * scores.std()), 2)

First, ridge regression with the zip code as 70 one-hot columns and the other numbers scaled. Run the cell. It takes a few seconds.

In [ ]:
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMBERS = [c for c in FEATURES if c not in ("zipcode", "lat", "long")]
prepare = ColumnTransformer([("zipcode", OneHotEncoder(handle_unknown="ignore"), ["zipcode"]),
                             ("numbers", StandardScaler(), NUMBERS)])
ridge = Pipeline([("prepare", prepare), ("model", Ridge(alpha=1.0))])
ridge_cv = cv_mape(ridge)
ridge_result = report(valid[TARGET], ridge.fit(train, train[TARGET]).predict(valid))
print("CV", ridge_cv, "| valid", ridge_result)

> **Check.** You should see `CV (19.53, 0.37)` and a valid MAPE of 21.3: worse than the zip-code baseline.

Now the same model on the **logarithm** of the price. `TransformedTargetRegressor` takes the log before `fit()` and turns the estimate back into dollars after `predict()`. On the log scale, a 10% miss is the same size for every house.

> **Predict.** Will the MAPE fall a little or a lot? Then run the cell.

In [ ]:
ridge_log = TransformedTargetRegressor(regressor=ridge, func=np.log, inverse_func=np.exp)
ridge_log_cv = cv_mape(ridge_log)
ridge_log_result = report(valid[TARGET], ridge_log.fit(train, train[TARGET]).predict(valid))
print("CV", ridge_log_cv, "| valid", ridge_log_result)

> **Check.** You should see `CV (13.55, 0.05)` and a valid MAPE of 14.9, with 0.486 within 10%.

Log both. Run the cell.

In [ ]:
log_experiment("E04", "Ridge, zip code one-hot, scaled numbers", ridge_cv, ridge_result, "reject",
               "Worse than the zip baseline on valid: its errors grow with the price.")
log_experiment("E05", "E04 on the log of the price", ridge_log_cv, ridge_log_result, "keep",
               "MAPE down by about 6 points: errors scale with the price.")
show_log()

Next, gradient boosting from [Ensembles](https://learning.nextia-ai.com/courses/ml/m05/ensembles/), on the log price. Trees can use latitude and longitude directly: they cut the map into small areas. So the boosting model uses `lat` and `long`, not the zip code. Run the cell. It takes about 15 seconds.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor

TREE_FEATURES = [c for c in FEATURES if c != "zipcode"]  # trees use lat and long for the place

def boosting(**settings):
    select = ColumnTransformer([("features", "passthrough", TREE_FEATURES)])
    model = Pipeline([("select", select), ("model", HistGradientBoostingRegressor(random_state=0, **settings))])
    return TransformedTargetRegressor(regressor=model, func=np.log, inverse_func=np.exp)

boosting_cv = cv_mape(boosting())
boosting_result = report(valid[TARGET], boosting().fit(train, train[TARGET]).predict(valid))
print("CV", boosting_cv, "| valid", boosting_result)

> **Check.** You should see `CV (11.71, 0.19)` and a valid MAPE of 13.2, with 0.554 within 10%.

A slower boosting (learning rate 0.05, 600 trees) is the obvious next step. Is its gain real? Run the cell.

In [ ]:
slower_cv = cv_mape(boosting(learning_rate=0.05, max_iter=600))
print("E06", boosting_cv, "| E07", slower_cv)
standard_error = slower_cv[1] / np.sqrt(5)  # the std of the 5 fold scores / the square root of 5
print("limit = best mean + one standard error =", round(slower_cv[0] + standard_error, 2))

> **Check.** You should see `E06 (11.71, 0.19) | E07 (11.57, 0.24)` and `limit = best mean + one standard error = 11.68`.

**What this tells us.** Here a lower score is better, so the [one-standard-error rule](https://learning.nextia-ai.com/courses/ml/m06/tune-hyperparameters/#the-one-standard-error-rule) adds the standard error to the best mean. E06 (11.71) is just above the limit (11.68), so it does not tie with E07: keep E07. Run the cell. It checks E07 on valid and logs both.

In [ ]:
slower_result = report(valid[TARGET], boosting(learning_rate=0.05, max_iter=600).fit(train, train[TARGET]).predict(valid))
print("E07 on valid:", slower_result)
log_experiment("E06", "Gradient boosting on the log price, lat and long, defaults", boosting_cv, boosting_result,
               "reject", "1.8 points better than E05 in CV, but just outside the limit of E07.")
log_experiment("E07", "E06 with learning_rate 0.05 and 600 trees", slower_cv, slower_result, "keep",
               "Best CV; E06 is above the one-standard-error limit. Valid agrees.")
show_log()

### Your turn: a model without the place

How much of the error does the place remove?

> **Your turn.** Set `TREE_FEATURES` to the features without `zipcode`, `lat` and `long`. Compute `no_place_cv` with `cv_mape(boosting())`, and `no_place_result` with `report()` on valid. Log it as E08 ("E06 without lat and long"), then set `TREE_FEATURES` back to the features without `zipcode`. Run the check cell.
>
> **Predict** first: how many points will the MAPE go up?

In [ ]:
TREE_FEATURES = ...  # the features without zipcode, lat and long
no_place_cv = ...  # cv_mape(boosting())
no_place_result = ...  # report() on valid
# log_experiment("E08", ..., no_place_cv, no_place_result, ..., ...)
TREE_FEATURES = [c for c in FEATURES if c != "zipcode"]  # back to the E06 features

In [ ]:
expect_hash('no_place_result', no_place_result, '7de926614cdde18c')

Without the place, the valid MAPE is almost twice as large (23.9 against 13.2 for E06). The place is a large part of the price.

## 5. Failure case: the random split

Chris, from the loans team, would split at random, as in many tutorials. Run the cell.

In [ ]:
from sklearn.model_selection import train_test_split

shuffled_train, shuffled_test = train_test_split(houses, test_size=0.2, random_state=0)
model = boosting().fit(shuffled_train, shuffled_train[TARGET])
print("random split:", report(shuffled_test[TARGET], model.predict(shuffled_test)))
twins = shuffled_test["id"].isin(shuffled_train["id"])
print("test sales of a house that is also in train:", int(twins.sum()))

> **Check.** You should see a MAPE of 11.9 and `test sales of a house that is also in train: 53`.

**What this tells us.** The random split looks better (11.9 against 13.2 for the same model) because it mixes the months: the model learns the spring market and is tested on it. Nadia's model must estimate next month's sales from last month's market. The random split also puts 53 sales of the same houses on both sides.

## 6. Metric and operating point (on valid)

The chosen model, E07, on valid. Run the cell.

In [ ]:
chosen = boosting(learning_rate=0.05, max_iter=600).fit(train, train[TARGET])
valid_estimate = chosen.predict(valid)
print(report(valid[TARGET], valid_estimate))

> **Check.** You should see a MAPE of 12.9, 0.556 within 10% and 0.194 over by 10% or more.

Errors by the **estimated** price band (the real price is not known when the estimate is made). Run the cell.

In [ ]:
BANDS = [0, 400_000, 700_000, 1_000_000, np.inf]
BAND_NAMES = ["under 400k", "400k-700k", "700k-1M", "over 1M"]

def slice_table(part, estimate, by):
    pct = 100 * (part[TARGET] - estimate) / part[TARGET]
    errors = part.assign(pct=pct, abs_pct=pct.abs(), within=pct.abs() <= 10)
    return errors.groupby(by, observed=True).agg(sales=("pct", "size"), median_error=("pct", "median"),
                                                 MAPE=("abs_pct", "mean"), within_10=("within", "mean")).round({"median_error": 1, "MAPE": 1, "within_10": 3})

estimate_band = pd.cut(valid_estimate, BANDS, labels=BAND_NAMES)
print(slice_table(valid, valid_estimate, estimate_band).to_string())

> **Check.** For estimates over 1 million you should see 124 sales, a MAPE of 17.1 and 0.395 within 10%. The other bands are between 0.519 and 0.653.

**What this tells us.** Estimates above 1 million dollars are the weakest band. So Nadia's rule: use the estimate only when it is **under 1 million dollars**; above that, an appraiser starts at once. This limit is the operating point. Run the cell.

In [ ]:
auto = valid_estimate < 1_000_000
print("share of valid sales with an estimate under 1M:", round(float(auto.mean()), 3))
print("those sales:", report(valid[TARGET][auto], valid_estimate[auto]))
print("the others: ", report(valid[TARGET][~auto], valid_estimate[~auto]))

> **Check.** You should see 0.943 of valid sales under 1 million, with 0.566 within 10%; the others have 0.395.

> **Your turn.** Nadia asks: what if the limit were 700,000 dollars? Compute the share of valid sales with an estimate under 700,000 and their `report()`. Put the report in `policy_700k`, then run the check cell.

In [ ]:
under_700k = ...  # True where the estimate is under 700,000
policy_700k = ...  # report() for those sales

In [ ]:
expect_hash('policy_700k', policy_700k, 'ad538379d307e730')

The lower limit keeps the share within 10% at about 0.555, and sends more houses to the appraiser. Keep the limit of 1 million.

## 7. Errors and slices (on valid)

> **Your turn.** Make the same table as for the price bands, by area: call `slice_table()` with `valid["area"]`. Put it in `area_slices`, then run the check cell.

In [ ]:
area_slices = ...  # slice_table() of valid, by area

In [ ]:
expect_hash('area_slices', area_slices, 'da71fb12700b6286')

Seattle has the most errors: 0.474 within 10%, against about 0.6 in the other areas.

The largest misses. Run the cell. The first three are the largest over-estimates, the last three the largest under-estimates.

In [ ]:
check = valid.assign(estimate=valid_estimate.round(-3), error_pct=(100 * (valid[TARGET] - valid_estimate) / valid[TARGET]).round(0))
columns = ["date", "price", "estimate", "error_pct", "sqft_living", "grade", "condition", "yr_built", "area"]
print(check.sort_values("error_pct")[columns].head(3).to_string(index=False))
print(check.sort_values("error_pct")[columns].tail(3).to_string(index=False))

> **Check.** At the top you should see a sale of 130,000 dollars with an estimate of 416,000.

**What this tells us.** Some sales are not normal market sales (a sale inside a family, a forced sale), and some old houses are in a state that the columns do not show. A human appraiser sees both; the model cannot.

## 8. The final test, once

The plan was fixed before this cell: refit E07 on train and valid, then estimate the test sales once. Run the cell.

In [ ]:
train_and_valid = pd.concat([train, valid])
final = boosting(learning_rate=0.05, max_iter=600).fit(train_and_valid, train_and_valid[TARGET])
test_estimate = final.predict(test)
final_result = report(test[TARGET], test_estimate)
print("final model on test:", final_result)

> **Check.** You should see `final model on test: {'median error %': 6.4, 'MAPE %': 12.8, 'within 10%': 0.502, 'over by 10%+': 0.126}`.

The zip-code baseline, learned the same way. Run the cell.

In [ ]:
zip_per_sqft = (train_and_valid[TARGET] / train_and_valid["sqft_living"]).groupby(train_and_valid["zipcode"]).median()
print("zip price per sq ft:", report(test[TARGET], test["zipcode"].map(zip_per_sqft) * test["sqft_living"]))

> **Check.** You should see a MAPE of 18.5 and 0.349 within 10%.

The slices of the test months: we report them, we do not tune on them. Run the cell.

In [ ]:
auto = test_estimate < 1_000_000
print("estimate under 1M:", round(float(auto.mean()), 3), report(test[TARGET][auto], test_estimate[auto]))
print(slice_table(test, test_estimate, test["area"]).to_string())
print(slice_table(test, test_estimate, test["date"].dt.month).to_string())

> **Check.** You should see the median error grow from 5.1% in March to 8.3% in May.

**What this tells us.** The MAPE stayed at about 13%, but the estimates are now **too low** by about 6%: the spring market rose after the model's data ended. Within 10% fell from 0.556 to 0.502. A price model must follow the market.

## 9. Save the model and the log

Run the cell. It saves the final model to `house_price_model.joblib` and shows the log.

In [ ]:
import joblib

joblib.dump(final, "house_price_model.joblib")
show_log()

> **Check.** You should see eight rows, E01 to E08.

## Next

The [lesson page](https://learning.nextia-ai.com/courses/ml/m09/estimate-house-prices/) has what a 13% error means for a lender, the recommendations for Nadia, the limits, the model card, how to make this a portfolio piece, and the knowledge checks.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Predict credit-card default](https://learning.nextia-ai.com/courses/ml/m09/predict-credit-card-default/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/ml/m09/estimate-house-prices/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The data is real: King County house sales 2014-2015 (OpenML 42092), CC0.